# Lab 05: Hooks & Guardrails

**Goal:** run your own code at points in the agent lifecycle to observe, limit, or block behavior, including human-in-the-loop approval with interrupts.

Course module: `samples/05-hooks`

Common events (in `strands.hooks`): `BeforeInvocationEvent`, `AfterInvocationEvent`, `BeforeModelCallEvent`, `AfterModelCallEvent`, `BeforeToolCallEvent`, `AfterToolCallEvent`, `MessageAddedEvent`.

Callback handlers (Lab 04) are for **displaying** output. Hooks are for **changing behavior**.

In [ ]:
# Run this first in every lab: puts the repo root on sys.path and loads lab_config
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from lab_config import get_model, describe
print(describe())

## 1. An observability hook
A `HookProvider` registers callbacks for the events it cares about. This one times every tool call.

In [ ]:
from strands import Agent, tool
from strands.hooks import (
    HookProvider, HookRegistry,
    BeforeInvocationEvent, BeforeToolCallEvent, AfterToolCallEvent,
)

@tool
def lookup_weather(city: str) -> str:
    """Get the current weather for a city.

    Args:
        city: Name of the city
    """
    fake = {"seattle": "52°F, rain", "austin": "88°F, sunny", "denver": "61°F, windy"}
    return f"{city}: {fake.get(city.lower(), '70°F, clear')}"


class ToolLogger(HookProvider):
    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(AfterToolCallEvent, self.log)

    def log(self, event: AfterToolCallEvent) -> None:
        tu = event.tool_use
        print(f"📋 hook: {tu['name']}({tu['input']}) → {event.result['status']}" + (f" in {event.duration:.3f}s" if event.duration else ""))


agent = Agent(model=get_model(), tools=[lookup_weather], hooks=[ToolLogger()], callback_handler=None)
print(agent("Compare the weather in Seattle and Austin."))

## 2. A guardrail: limit tool calls per request
From `rate_limiter.py`. Setting `event.cancel_tool` to a string **skips the tool** and returns that string to the model as the tool's result, so the model can adapt.

In [ ]:
class LimitToolCounts(HookProvider):
    def __init__(self, max_calls: int = 3):
        self.max_calls = max_calls
        self.counts: dict[str, int] = {}

    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(BeforeInvocationEvent, self.reset)   # fresh budget per request
        registry.add_callback(BeforeToolCallEvent, self.check)

    def reset(self, event: BeforeInvocationEvent) -> None:
        self.counts = {}

    def check(self, event: BeforeToolCallEvent) -> None:
        name = event.tool_use["name"]
        self.counts[name] = self.counts.get(name, 0) + 1
        if self.counts[name] > self.max_calls:
            print(f"⛔ blocked call #{self.counts[name]} to {name}")
            event.cancel_tool = f"'{name}' hit the {self.max_calls}-call limit. Do NOT call this tool again."


agent = Agent(model=get_model(), tools=[lookup_weather],
              hooks=[ToolLogger(), LimitToolCounts(max_calls=2)], callback_handler=None)
print(agent("Get the weather for Seattle, Austin, Denver, and Boston, one call per city."))

## 3. Human-in-the-loop with interrupts
From `approval_interrupt.py`. `event.interrupt(name, reason=...)` **pauses the agent**. The call returns with `stop_reason == "interrupt"`. You collect a decision and resume by sending an `interruptResponse`.

Set `INTERACTIVE = True` to be prompted with `input()`. Otherwise the cell auto-answers with `AUTO_ANSWER`.

In [ ]:
import os, pathlib

INTERACTIVE = False
AUTO_ANSWER = "n"   # try "y" too

sandbox = pathlib.Path("workspace/hooks_demo"); sandbox.mkdir(parents=True, exist_ok=True)
for name in ["keep.txt", "old_notes.txt"]:
    (sandbox / name).write_text(f"contents of {name}\n")

@tool
def list_files(directory: str) -> str:
    """List files in a directory.

    Args:
        directory: Path to the directory
    """
    try:
        return "\n".join(os.listdir(directory)) or f"{directory} is empty"
    except FileNotFoundError:
        return f"Directory not found: {directory}"

@tool
def delete_file(path: str) -> str:
    """Delete a file.

    Args:
        path: Path to the file to delete
    """
    try:
        os.remove(path); return f"Deleted {path}"
    except FileNotFoundError:
        return f"File not found: {path}"


class DeleteApprovalHook(HookProvider):
    """Only intercepts deletes. Other tools run freely."""
    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(BeforeToolCallEvent, self.check_delete)

    def check_delete(self, event: BeforeToolCallEvent) -> None:
        if event.tool_use["name"] != "delete_file":
            return
        approval = event.interrupt("delete-approval", reason={"path": event.tool_use["input"]["path"]})
        if str(approval).lower() != "y":
            event.cancel_tool = "User denied file deletion"

In [ ]:
agent = Agent(model=get_model(), tools=[list_files, delete_file],
              hooks=[DeleteApprovalHook()], callback_handler=None)

result = agent(f"Delete old_notes.txt from {sandbox.resolve()}")

while result.stop_reason == "interrupt":
    responses = []
    for intr in result.interrupts:
        prompt = f"⚠️  Delete '{intr.reason['path']}'? (y/N): "
        answer = input(prompt) if INTERACTIVE else AUTO_ANSWER
        print(prompt + answer)
        responses.append({"interruptResponse": {"interruptId": intr.id, "response": answer}})
    result = agent(responses)

print("\n🤖", str(result).strip())
print("files now:", sorted(os.listdir(sandbox)))

## 🧪 Exercise
Write a `BeforeToolCallEvent` hook that blocks `delete_file` for any path outside `workspace/`, with no human prompt. Test it by asking the agent to delete `/tmp/something`.

**Next:** [Lab 06: Sessions & Persistent Memory](06-sessions.ipynb)

In [ ]:
# your code here